# MedCLIP Walkthrough

Contrastive learning from unpaired medical images and text (EMNLP 2022).

In [ ]:
import os
import sys
import torch
import yaml
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

sys.path.append("..")
from src.model import MedCLIP
from src.dataset import MedCLIPDataset
from src.losses import SemanticMatchingLoss

In [ ]:
with open("../configs/default.yaml", "r") as f:
    config = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MedCLIP(config).to(device)
model.eval()
print(f"Model loaded on {device}")

In [ ]:
data_root = os.environ.get("MIMIC_CXR_ROOT")
dataset = MedCLIPDataset(
    data_root=data_root,
    split="test",
    concept_file=config.get("concept_file", "concepts.json"),
    image_size=config["image_size"],
    max_length=config["max_text_length"],
)
loader = DataLoader(dataset, batch_size=2, shuffle=True)
batch = next(iter(loader))
print("Batch keys:", batch.keys())

In [ ]:
images = batch["image"].to(device)
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)

with torch.no_grad():
    img_feats, txt_feats = model(images, input_ids, attention_mask)

print(f"Image features shape: {img_feats.shape}")
print(f"Text features shape: {txt_feats.shape}")

In [ ]:
loss_fn = SemanticMatchingLoss(margin=config["margin"], hard_negative_mining=True)
loss = loss_fn(img_feats, txt_feats, batch["concept_labels"].to(device), threshold=config["concept_threshold"])
print(f"Semantic matching loss: {loss.item():.4f}")

In [ ]:
sim = img_feats @ txt_feats.T
plt.figure(figsize=(6,5))
plt.imshow(sim.cpu(), cmap='viridis', aspect='auto')
plt.colorbar(label='Cosine similarity')
plt.xlabel('Text index')
plt.ylabel('Image index')
plt.title('Image-Text similarity matrix')
plt.show()